# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a complete, step-by-step guide for loading and exploring the FAIR\(^2\) dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library. All entities are referenced by their `@id` in accordance with best practices for Croissant datasets.

### Dataset Source
* Croissant metadata: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

* Citation: Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

In [ ]:
# Ensure `mlcroissant` and requirements are installed
!pip install mlcroissant --quiet

## 1. Data Loading

We begin by loading the Croissant dataset metadata and records using `mlcroissant`. The dataset is referenced by its schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant JSON-LD metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get the dataset metadata object (CroissantDataset)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")
print(f"Version: {metadata.version}\n")

## 2. Data Overview

Review available record sets, their fields, and associated column IDs. All entities are referenced by their `@id`.

Let's examine the record sets contained in the dataset and view their IDs and basic structure.

In [ ]:
print("Available record sets (by @id):")
record_set_ids = [rset['@id'] for rset in metadata.json_dict.get('recordSet', [])]
for idx, rset in enumerate(metadata.json_dict.get('recordSet', [])):
    print(f"  {idx+1}. @id = {rset['@id']} | name = {rset.get('name', '<no name>')}")
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # Each field is a reference (by @id)
        print(f"      field @id: {field['@id']}" if isinstance(field, dict) else f"      field @id: {field}")
if not record_set_ids:
    print('No record sets found. This dataset metadata does not expose record sets in root JSON-LD (may require inspection or manual entry).')

### Attempting to enumerate record sets directly

**Note:** If the previous cell returned no record sets, the Croissant schema does not enumerate them under the `recordSet` field by default. For some datasets, record sets are linked only through the files/distributions. We'll fetch and list available record set IDs directly from the Dataset object as well.

In [ ]:
# Use the convenience Dataset API to enumerate record sets
all_record_set_ids = list(dataset.get_record_sets())
print('Record sets as reported by mlcroissant:')
for rsid in all_record_set_ids:
    print(' -', rsid)

# In this dataset, the main record set ID is likely 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034' (from the distribution field)
# but let's confirm by printing the first N rows from all detected record sets.
if not all_record_set_ids:
    print('No record sets found via API either: check Croissant schema for correct @id references.')

Let's display a few records from each record set (referenced by `@id`):

In [ ]:
# Print some sample records for the record sets found (by @id)
num_preview = 2
for rsid in all_record_set_ids:
    print(f"\nRecord set @id: {rsid} - Example records:")
    try:
        records_iter = dataset.records(record_set=rsid)
        for i, rec in enumerate(records_iter):
            print(f"  Row {i+1}: {json.dumps(rec, indent=2)[:350]}")
            if i+1 >= num_preview:
                break
    except Exception as e:
        print(f"  Could not load records for {rsid}: {e}")

## 3. Data Extraction

Load data from the main record set into a pandas DataFrame for analysis. Record set and columns are referenced by their `@id`.

We'll select the primary record set (usually the first or the largest), and extract the records to DataFrames, indexed by their record set `@id`.

In [ ]:
# Prepare to load records into DataFrames
dataframes = {}
# Use the main record set (if available)
record_sets_to_load = all_record_set_ids
for rsid in record_sets_to_load:
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            dataframes[rsid] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set @id: {rsid}")
            print(f"Columns (@id):\n", dataframes[rsid].columns.tolist())
            print(dataframes[rsid].head(2))
        else:
            print(f"No records found for {rsid}")
    except Exception as e:
        print(f"Error loading {rsid}: {e}")
# Pick the main table's record set for further analysis
main_rsid = record_sets_to_load[0] if record_sets_to_load else None

## 4. Exploratory Data Analysis (EDA)

Now that we've loaded the main record set into a DataFrame, let's apply common EDA steps such as filtering, normalization, and grouping.

> **Tip:** The below code assumes that the column (field) of interest has a recognizable numeric name, such as `Coverage (%)`, `MW [Da]`, `Peptides (All)`, etc. All operations reference columns by their field `@id` as given in the dataframe.

In [ ]:
# Find a numeric field by its @id (column name in the DataFrame)
if main_rsid and main_rsid in dataframes:
    df = dataframes[main_rsid]
    # Try to auto-detect a numeric column
    sample_numeric_fields = [c for c in df.columns if df[c].dtype.kind in 'fi']
    if sample_numeric_fields:
        numeric_field_id = sample_numeric_fields[0]
        print(f"Using numeric field for EDA: {numeric_field_id}")
    else:
        print("No directly numeric columns found. Trying to convert...")
        for c in df.columns:
            # Try parsing as float if possible
            try:
                df[c+'_float'] = pd.to_numeric(df[c], errors='coerce')
                if df[c+'_float'].notna().sum() > 0:
                    numeric_field_id = c+'_float'
                    print(f"Using numeric field (parsed): {numeric_field_id} (original: {c})")
                    break
            except Exception:
                continue
        else:
            print("No usable numeric fields found! Aborting EDA section.")
            numeric_field_id = None

    # Continue EDA if numeric field available
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fi' else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} ({len(filtered_df)} rows):")
        print(filtered_df.head(3))

        # Normalize this numeric field
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head(3))

        # Find a likely grouping field (categorical, e.g., 'Description' or sample name)
        group_field_id = None
        for c in df.columns:
            if c != numeric_field_id and df[c].dtype == 'object' and df[c].nunique() < len(df) and df[c].nunique() < 20:
                group_field_id = c
                print(f'Grouping by field: {group_field_id}')
                break

        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped means by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("Unable to continue EDA without a valid numeric field.")
else:
    print("No main record set DataFrame loaded. Please check data extraction.")

## 5. Visualization

Visualize the distribution of the chosen numeric field and the effect of normalization, grouping, etc. All field references use their `@id` values as in previous steps.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if main_rsid and main_rsid in dataframes and 'numeric_field_id' in locals() and numeric_field_id:
    df = dataframes[main_rsid]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If normalized version exists from filtering
    if numeric_field_id+"_normalized" in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field_id+"_normalized"].dropna(), bins=30, color='g', kde=True)
        plt.title(f"Normalized distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id+"_normalized")
        plt.show()
else:
    print("Visualization impossible: no numeric field detected or dataframe invalid.")

## 6. Conclusion

In this notebook, we demonstrated how to load, explore, and process a FAIR\(^2\) Croissant dataset using the `mlcroissant` library, referencing all dataset parts by their unique `@id`. We:
- Loaded both metadata and tabular records;
- Programmatically explored available record sets, fields, and columns;
- Extracted record set data using field `@id`s;
- Carried out basic EDA, normalization, and grouping;
- Visualized key numeric distributions.

You can extend this notebook by selecting other record sets (using their `@id`), analyzing additional fields, or building advanced visualizations. For more information, consult the [mlcroissant documentation](https://mlcommons.org/croissant/).

> **Provenance:** All references to record sets, columns, and fields are by their exact Croissant `@id` as per best-practice reproducibility.